Import relevant libraries 

In [ ]:
%pip install numpy pandas matplotlib 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Configure pandas display settings.
Display all DataFrame columns without truncation and format floating-point values with comma separators and two decimal places. 

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

Set project root 

In [ ]:
current_directory = Path.cwd()
if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

print("Project root:" f"{project_root}")

Read the csv data using the project root and its subsequent folders 

In [ ]:
data_path = (
    project_root
    / "data"
    / "raw"
    / "credit_default_uci.csv"
)

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Perform initial validity checks to check that:
- The dataset contains the expected 30,000 rows and 25 columns.
- Every customer ID is unique.
- The target variable contains only the expected binary values: `0` for no default and `1` for default.

In [ ]:
assert df.shape == (30000, 25)
assert df["customer_id"].is_unique
assert set(df["default_next_month"].unique()) == {0, 1}

print("Initial validity checks passed")

Create maps for category codes in the educaion column.
Convert the numerical education codes into interpretable category labels and store them in a new 'education_group' column. 
Unrecorded category codes will be mapped to unknown. 
Cross-tabulation is then used to verify how each original education code maps to the new grouped categories.

In [ ]:
education_map = {
    1 : "Graduate school",
    2 : "University",
    3 : "Highschool",
    4 : "Other"
}

df["education_group"] = (df["education"]
.map(education_map)
.fillna("Unknown")
                        )

pd.crosstab(
    df["education"],
    df["education_group"]
)

Create maps for category codes in the marriage column.
Convert the numerical marriage codes into interpretable category labels and store them in a new 'marriage_group' column. 
Unrecorded category codes will be mapped to unknown. 
Cross-tabulation is then used to verify how each original marriage code maps to the new grouped categories.

In [ ]:
marriage_map = {
    1: "Married",
    2: "Single",
    3: "Other",
}

df["marriage_group"] = (
    df["marriage"]
    .map(marriage_map)
    .fillna("Unknown")
)

pd.crosstab(
    df["marriage"],
    df["marriage_group"]
)

Combine the six monthly repayment-status columns into a single series and count how frequently each status code occurs across all customer-month records. Sort the results by repayment-status code.
These frequencies represent monthly observations rather than unique customers.

In [ ]:
repayment_columns = [
    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",
]

repayment_code_counts = (df[repayment_columns]
                     .stack()
                     .value_counts()
                     .sort_index()
                    )

repayment_code_counts

Count how many of the six recorded months have a positive repayment-status code for each customer. Positive codes indicate a recorded payment delay, while zero and negative codes are treated as no recorded delay.

In [ ]:
df['months_with_recorded_delay'] = (
    df[repayment_columns] > 0).sum(axis=1)
df['months_with_recorded_delay'].value_counts().sort_index()

Replace negative repayment-status codes with zero and identify the highest recorded delay across the six months for each customer.

In [ ]:
df['maximum_recorded_delay'] = df[repayment_columns].clip(lower=0).max(axis=1)
df['maximum_recorded_delay'].value_counts().sort_index()

Categorise customers according to their September repayment status, which is the most recent month in the dataset. Customers are grouped into those with no recorded positive delay, those one month late, and those two or more months late.

In [ ]:
df["recent_repayment_group"] = pd.cut(
    df["repayment_status_sep"],
    bins = [-np.inf, 0, 1, np.inf],
    labels = ["No recorded delay",
              "One month late",
              "Two or more months late"]
)

df["recent_repayment_group"].value_counts(dropna = False)

In [ ]:
overall_default_rate = df["default_next_month"].mean()

print(f"Overall default rate: {overall_default_rate:.2%}")

Customers are grouped according to their most recent repayment status to examine
whether payment delays are associated with default in the following month.

For each repayment group, the analysis calculates:

- Customers: total number of customers in the group
- Defaults: number of customers who defaulted
- Default rate: proportion of customers who defaulted

In [ ]:
repayment_summary = (df.groupby(
    df["recent_repayment_group"],
    observed = False
)
["default_next_month"]
.agg(
    customers = "size",
    defaults =  "sum",
    default_rate = "mean"
)
.reset_index()
)

repayment_summary

Add a percentage column for the default rate.

In [ ]:
repayment_summary["default_rate_percentage"] = (
    repayment_summary["default_rate"] * 100
)

repayment_summary

Compare the default rate across the three most recent repayment-status groups. The y-axis and bar labels display default rates as percentages to support straightforward comparison between customer groups.

In [ ]:
groups = repayment_summary["recent_repayment_group"].astype(str)
default_rates = repayment_summary["default_rate"]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    groups,
    default_rates,
    color=["#4C78A8", "#F2CF5B", "#E45756"]
)

plt.title("Default Rate by Most Recent Repayment Status")
plt.xlabel("Repayment status")
plt.ylabel("Default rate")

plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

plt.bar_label(
    bars,
    labels=[f"{rate:.1%}" for rate in default_rates],
)


repayment_figure_path = (
    project_root
    / "outputs"
    / "figures"
    / "default_rate_by_repayment_status.png"
)

plt.savefig(
    repayment_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Customers are divided into four approximately equal-sized groups based on
their credit limits. Using quartiles makes it possible to compare default
rates between customers with relatively low and high limits without assuming
that the relationship is linear.

In [ ]:
df["credit_limit_band"] = pd.qcut(
    df["credit_limit"],
    q = 4,
    labels = ["Lowest 25%",
              "Lower-middle 25%",
              "Upper-middle 25%",
              "Highest 25%",
             ]
)

The minimum, maximum and median credit limit within each band are examined
before calculating its default rate.

In [ ]:
credit_limit_ranges = (
    df.groupby(df["credit_limit_band"], observed = False)
    ["credit_limit"]
    .agg(["count", "min", "max", "median"])
    .reset_index()
)

credit_limit_ranges

A credit limit can reflect both creditworthiness and financial exposure.

A higher limit may indicate that the lender previously considered the
customer less risky. However, it also increases the amount that the lender
could potentially lose.

Consequently, an association between higher limits and lower default rates
would not prove that increasing a customer's limit reduces their risk. The
original lender may simply have assigned higher limits to customers who
already appeared more creditworthy.

In [ ]:
credit_limit_summary = (
    df.groupby(
        "credit_limit_band",
        observed=False
    )["default_next_month"]
    .agg(
        customers="size",
        defaults="sum",
        default_rate="mean",
    )
    .reset_index()
)

credit_limit_summary["default_rate_percentage"] = (
    credit_limit_summary["default_rate"] * 100
)

credit_limit_summary

The following chart compares default rates across the four credit-limit
quartiles. It helps identify whether customers with relatively low or high
limits show different subsequent default patterns.

These comparisons are descriptive and should not be interpreted as evidence
that changing a customer's credit limit would cause their default risk to
change.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    credit_limit_summary["credit_limit_band"].astype(str),
    credit_limit_summary["default_rate"],
    color="#4C78A8",
)

ax.set_title("Default Rate by Credit-Limit Quartile")
ax.set_xlabel("Credit-limit group")
ax.set_ylabel("Default rate")

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda value, _: f"{value:.0%}")
)

for bar in bars:
    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.1%}",
        ha="center",
        va="bottom",
    )

plt.xticks(rotation=15)
plt.tight_layout()

credit_limit_figure_path = (
    project_root
    / "outputs"
    / "figures"
    / "default_rate_by_credit_limit.png"
)

plt.savefig(
    credit_limit_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Summary statistics are examined for the customer's credit limit, most recent
bill balance and most recent payment amount.

The 1st and 99th percentiles are included to assess skewness and identify
extreme observations. Large differences between the median, mean, 99th
percentile and maximum may indicate highly skewed financial variables.

Extreme values are not automatically removed because they may represent
genuine customers with unusually large balances, limits or payments.

In [ ]:
recent_financial_summary = df[
    [
        "credit_limit",
        "bill_amount_sep",
        "payment_amount_sep",
    ]
].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

recent_financial_summary

Default rates are compared across sex, education and marital-status groups for
data-understanding and fairness-auditing purposes.

Observed differences do not establish that demographic characteristics cause
default. These variables may also raise ethical, fairness and regulatory
concerns if used directly in lending decisions. Therefore, demographic
associations will not automatically be used to justify approval or decline
decisions.

In [ ]:
def create_default_summary(data, column_name):
    summary = (
        data.groupby(column_name, observed = False)["default_next_month"]
        .agg(
        customer = "size",
        defaults = "sum",
        default_rate = "mean"
        ).reset_index()
    )

    summary["default_rate_percentage"] = summary["default_rate"] * 100
    return summary

In [ ]:
sex_map = {
    1: "Male",
    2: "Female"
}

df["sex_group"] = (
    df["sex"]
    .map(sex_map)
    .fillna("Unknown")
)

In [ ]:
sex_summary = create_default_summary(df, "sex_group")
education_summary = create_default_summary(df, "education_group")
marriage_summary = create_default_summary(df, "marriage_group")

display(sex_summary)
display(education_summary)
display(marriage_summary)

In [ ]:
repayment_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "repayment_risk_summary.csv",
    index=False,
)

credit_limit_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "credit_limit_risk_summary.csv",
    index=False,
)

recent_financial_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "recent_financial_summary.csv"
)

Save the cleaned customer-level dataset, including the categories and repayment
features created during Day 2. This file will be used for feature engineering
in the next notebook.

In [ ]:
processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "credit_default_cleaned.csv"
)

df.to_csv(processed_data_path, index=False)

print(f"Cleaned dataset saved to: {processed_data_path}")

## Day 2 findings

The overall subsequent default rate was 22.12%.

Customers with a recent recorded repayment delay had a higher
default rate than customers without a positive recorded delay. The default
rate was 13.83% among customers with no positive delay, compared with
69.55% among customers two or more months late.

Default rates also varied across credit-limit quartiles. Customers in the
lowest 25% had the highest observed default rate, while those in the highest 25% had
the lowest rate. This is an association and does not establish that changing
a customer's credit limit would cause their default risk to change.

Undocumented education and marital-status codes were labelled as "Unknown"
while the original variables were retained. No records were removed.

Demographic variables were examined for data-quality and fairness purposes,
but differences between demographic groups will not automatically be treated
as justification for lending decisions.

The next stage will engineer features representing utilisation, repayment
consistency and recent financial behaviour.